# MW-HQNet — Multi-Window Enhanced Hierarchical Query-Graph Network for small renal lesions (Kaggle notebook)

**What this notebook does (one pass, resumable):**
1. Streams **KiTS23** case by case from Hugging Face (`MedOtter/kits23`, per-case NIfTI), crops each kidney to a 192×192 ROI at ~1.0×1.0×2.5 mm, deletes the raw file. Re-running skips finished cases.
2. Builds 5-fold **patient-level** splits.
3. Trains **MW-HQNet** (PAWE enhancement → ResNet34 + AIFI hybrid encoder → U-Net decoder → lesion-centre query selection → hierarchical query-graph decoder → dense/query fusion). Checkpoints every epoch; resumes after a session ends.
4. Evaluates in 3D: Dice / NSD / HD95 for KiTS hierarchical classes, lesion-level sensitivity by size bin, false positives per kidney, low-dose robustness.
5. Ablations are config switches (`ENHANCE`, `QUERY`, `GRAPH`, `AIFI`, `FUSION`) — re-run with a different `RUN_NAME`.

**Settings:** Accelerator = GPU (T4 or P100), Internet = ON. For the stone track set `DATASET='kssd2025'` and attach the Kaggle dataset `murillobouzon/kssd2025-kidney-stone-segmentation-dataset`.

**First run:** set `SMOKE_TEST=True` (synthetic data, ~3 min) to verify the environment, then set it back to `False`.

Evaluation is in the cropped-ROI space with ROI taken from the ground-truth kidney mask (oracle localisation). Report it as such; Phase 2 of the roadmap adds a predicted-ROI stage.

**Optional:** add a Kaggle secret `HF_TOKEN` (Add-ons → Secrets) if Hugging Face rate-limits anonymous downloads; Cell 1 exports it.
**Kaggle sessions:** preprocessing takes roughly 30–60 min for all 489 cases. After it finishes, save the notebook version and add its output (`kits23_roi`) as a private dataset, then point `DATA_DIR` at it and set `RUN_PREPROCESS=False` for training sessions.
**KSSD2025:** TIFFs carry no voxel spacing, so "mm" in its metrics means pixels.

In [ ]:
# ===== Cell 1: installs (pure Python, no shell magics) =====
import importlib, subprocess, sys
def _ensure(pkg, pipname=None):
    try:
        importlib.import_module(pkg)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pipname or pkg], check=True)
for _p, _n in [("nibabel", None), ("timm", None), ("huggingface_hub", None), ("cv2", "opencv-python-headless"), ("pandas", None)]:
    _ensure(_p, _n)
try:  # optional Kaggle secret for Hugging Face downloads
    from kaggle_secrets import UserSecretsClient
    import os as _os
    _os.environ.setdefault("HF_TOKEN", UserSecretsClient().get_secret("HF_TOKEN"))
except Exception:
    pass
print("deps ok")

In [ ]:
# ===== Cell 2: configuration =====
import os, json, math, time, random, glob, gc, warnings
import numpy as np
warnings.filterwarnings("ignore")

ON_KAGGLE = os.path.exists("/kaggle")
ROOT = "/kaggle/working" if ON_KAGGLE else os.path.abspath("./work")

CFG = dict(
    DATASET="kits23",            # 'kits23' | 'kssd2025'
    SMOKE_TEST=False,            # True = synthetic data, fast end-to-end check
    RUN_PREPROCESS=False,        # KiTS23 streaming preprocessing (skips finished cases)
    HF_REPO="MedOtter/kits23",   # per-case NIfTI mirror of KiTS23 (CC BY-NC-SA)
    HF_REPO_FALLBACK="Angelou0516/kits23",
    N_CASES=None,                # None = all 489; e.g. 80 for a quick pilot
    PREP_WORKERS=4,
    ROI_PX=192, ROI_MM=192.0, Z_SPACING=2.5, XY_MARGIN_MM=16.0, Z_MARGIN_MM=10.0,
    N_SLICES=5,
    DATA_DIR="/kaggle/input/kits23-roi-192/kits23_roi",
    RAW_TMP="/tmp/kits_raw",
    KSSD_GLOB="/kaggle/input/**/*.tif*",
    OUT_DIR=os.path.join(ROOT, "runs"),
    RUN_NAME="baseline_unet",
    FOLD=0, N_FOLDS=5, SEED=42,
    EPOCHS=15, ITERS_PER_EPOCH=800, BATCH=12, LR=3e-4, WD=1e-4, WARMUP_EPOCHS=1,
    NUM_WORKERS=4, AMP=True, RESUME=True, VAL_EVERY=3, GRAD_CLIP=1.0,
    LESION_SAMPLE_P=0.6,         # probability of sampling a lesion-containing slice
    CROP=None,                   # None for KiTS (ROI already 192); 256 for KSSD training
    # ---- model / ablation switches ----
    ENHANCE="single",            # 'pawe' | 'pawe_nogate' | 'wso' (learned windows only) | 'fixed_multi' | 'single'
    AIFI=False,
    QUERY=False,                 # False = dense U-Net only (baseline)
    GRAPH="graph",               # 'graph' | 'selfattn' | 'none'
    FUSION=True, FUSION_BETA=0.5,
    N_QUERIES=32, D_MODEL=192, N_DEC_LAYERS=3, MASK_DIM=64, KNN=6,
    TF_P0=0.5,                   # teacher-forcing prob. of GT centres as queries (decays to 0 at 50% of training)
    W_QUERY=0.3, W_HM=1.0, W_TV=1e-3,
    PRETRAINED=True,
    # ---- augmentation ----
    LOWDOSE_P=0.3, BLUR_P=0.15, NOISE_SIGMA0=12.0, INT_SHIFT=20.0,
    # ---- evaluation ----
    TAU_HIGH=0.5, MIN_LESION_MM3=10.0, TEST_DOSES=(1.0, 0.5, 0.25),
)

if CFG["DATASET"] == "kits23":
    CFG.update(CLASSES=["bg", "kidney", "tumor", "cyst"], LESION_CLASSES=[2, 3], KIDNEY_CLASS=1,
               WINDOWS=[(40, 400), (30, 150), (150, 600), (400, 1800)], SINGLE_WINDOW=(40, 400),
               CE_WEIGHTS=[0.5, 1.0, 2.0, 2.0])
else:  # kssd2025: 8-bit images, windows in 0..255 units
    CFG.update(CLASSES=["bg", "stone"], LESION_CLASSES=[1], KIDNEY_CLASS=None,
               WINDOWS=[(128, 256), (170, 120), (210, 80), (240, 60)], SINGLE_WINDOW=(128, 256),
               CE_WEIGHTS=[0.5, 3.0], CROP=256, MIN_LESION_MM3=3.0, DATA_DIR=os.path.join(ROOT, "kssd_cache"),
               NOISE_SIGMA0=4.0, INT_SHIFT=8.0)

if CFG["SMOKE_TEST"]:
    CFG.update(DATA_DIR=os.path.join(ROOT, "synthetic_roi"), EPOCHS=2, ITERS_PER_EPOCH=6, BATCH=4,
               NUM_WORKERS=0, VAL_EVERY=1, RUN_NAME=CFG["RUN_NAME"] + "_smoke", PRETRAINED=False, N_FOLDS=3)

os.makedirs(CFG["DATA_DIR"], exist_ok=True) if not CFG["DATA_DIR"].startswith("/kaggle/input") else None
RUN_DIR = os.path.join(CFG["OUT_DIR"], f'{CFG["RUN_NAME"]}_fold{CFG["FOLD"]}')
os.makedirs(RUN_DIR, exist_ok=True)
json.dump(CFG, open(os.path.join(RUN_DIR, "config.json"), "w"), indent=1, default=str)

def seed_all(s):
    random.seed(s); np.random.seed(s)
    import torch; torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_all(CFG["SEED"])
print("RUN_DIR:", RUN_DIR)

In [ ]:
# ===== Cell 3: KiTS23 streaming preprocessing (resumable) =====
import nibabel as nib
from scipy import ndimage

def _side_ranges(fg, sx):
    """Split foreground along the canonical x-axis into left/right kidney ranges."""
    prof = fg.sum(axis=(1, 2)) > 0
    xs = np.where(prof)[0]
    if len(xs) == 0:
        return []
    x0, x1 = xs[0], xs[-1]
    gaps, run_start = [], None
    for x in range(x0, x1 + 1):
        if not prof[x] and run_start is None:
            run_start = x
        if prof[x] and run_start is not None:
            gaps.append((run_start, x - 1)); run_start = None
    if gaps:
        g = max(gaps, key=lambda t: t[1] - t[0])
        if (g[1] - g[0] + 1) * sx >= 10:
            mid = (g[0] + g[1]) // 2
            return [(x0, mid), (mid + 1, x1)]
    if (x1 - x0) * sx > 150:  # touching kidneys (rare): split at the middle
        mid = (x0 + x1) // 2
        return [(x0, mid), (mid + 1, x1)]
    return [(x0, x1)]

def _crop_resample(arr, spacing, center, out_px, out_sp, order, cval):
    """Sample a box centred at `center` (voxel coords) with out_px voxels of out_sp mm."""
    axes = []
    for a in range(3):
        g = (np.arange(out_px[a]) - (out_px[a] - 1) / 2.0) * out_sp[a] / spacing[a] + center[a]
        axes.append(g)
    lo = [max(int(np.floor(ax.min())) - 1, 0) for ax in axes]
    hi = [min(int(np.ceil(ax.max())) + 2, arr.shape[a]) for a, ax in enumerate(axes)]
    if any(h <= l for l, h in zip(lo, hi)):
        return np.full(out_px, cval, dtype=np.float32)
    sub = np.asarray(arr[lo[0]:hi[0], lo[1]:hi[1], lo[2]:hi[2]], dtype=np.float32)
    grid = np.meshgrid(axes[0] - lo[0], axes[1] - lo[1], axes[2] - lo[2], indexing="ij")
    return ndimage.map_coordinates(sub, grid, order=order, mode="constant", cval=cval, prefilter=False)

def preprocess_volume(img_nii, seg_nii, case_id, out_dir, cfg):
    img = nib.as_closest_canonical(img_nii); seg = nib.as_closest_canonical(seg_nii)
    sp = np.array(img.header.get_zooms()[:3], dtype=np.float64)
    hu = np.asanyarray(img.dataobj).astype(np.float32)
    lab = np.asanyarray(seg.dataobj).astype(np.uint8)
    fg = lab > 0
    saved = []
    for si, (xa, xb) in enumerate(_side_ranges(fg, sp[0])):
        side = np.zeros_like(fg); side[xa:xb + 1] = fg[xa:xb + 1]
        if side.sum() < 50:
            continue
        idx = np.argwhere(side)
        lo, hi = idx.min(0), idx.max(0)
        center = (lo + hi) / 2.0
        ext_mm = (hi - lo + 1) * sp
        roi_mm = max(cfg["ROI_MM"], max(ext_mm[0], ext_mm[1]) + 2 * cfg["XY_MARGIN_MM"])
        sxy = roi_mm / cfg["ROI_PX"]
        nz = int(math.ceil((ext_mm[2] + 2 * cfg["Z_MARGIN_MM"]) / cfg["Z_SPACING"]))
        out_px = (cfg["ROI_PX"], cfg["ROI_PX"], nz)
        out_sp = (sxy, sxy, cfg["Z_SPACING"])
        im = _crop_resample(hu, sp, center, out_px, out_sp, 1, -1024.0)
        sg = _crop_resample(lab, sp, center, out_px, out_sp, 0, 0.0)
        im = np.clip(np.round(im), -1024, 3071).astype(np.int16).transpose(2, 1, 0)  # (Z, Y, X)
        sg = sg.astype(np.uint8).transpose(2, 1, 0)
        key = f"{case_id}_s{si}"
        np.save(os.path.join(out_dir, key + "_img.npy"), im)
        np.save(os.path.join(out_dir, key + "_seg.npy"), sg)
        json.dump(dict(case=case_id, key=key, spacing=[cfg["Z_SPACING"], sxy, sxy]),
                  open(os.path.join(out_dir, key + "_meta.json"), "w"))
        saved.append(key)
    return saved

def _process_case(args):
    case_id, repo, cfg = args
    os.environ.setdefault("HF_HOME", "/tmp/hf_home")                 # keep HF caches off /kaggle/working
    os.environ.setdefault("HF_XET_CHUNK_CACHE_SIZE_BYTES", "0")       # no chunk cache: raw files are deleted after use
    from huggingface_hub import hf_hub_download
    done_flag = os.path.join(cfg["DATA_DIR"], f"{case_id}.done")
    if os.path.exists(done_flag):
        return case_id, "skip"
    tmp = os.path.join(cfg["RAW_TMP"], case_id); os.makedirs(tmp, exist_ok=True)
    try:
        token = os.environ.get("HF_TOKEN")
        pi = hf_hub_download(repo, f"dataset/{case_id}/imaging.nii.gz", repo_type="dataset", local_dir=tmp, token=token)
        ps = hf_hub_download(repo, f"dataset/{case_id}/segmentation.nii.gz", repo_type="dataset", local_dir=tmp, token=token)
        keys = preprocess_volume(nib.load(pi), nib.load(ps), case_id, cfg["DATA_DIR"], cfg)
        open(done_flag, "w").write(json.dumps(keys))
        return case_id, f"ok {len(keys)} side(s)"
    except Exception as e:
        return case_id, f"ERROR {type(e).__name__}: {e}"
    finally:
        import shutil; shutil.rmtree(tmp, ignore_errors=True)

def list_kits_cases(repo):
    from huggingface_hub import HfApi
    files = HfApi().list_repo_files(repo, repo_type="dataset", token=os.environ.get("HF_TOKEN"))
    return sorted({f.split("/")[1] for f in files if f.startswith("dataset/case_") and f.endswith("imaging.nii.gz")})

def make_synthetic_data(out_dir, n_cases=6, seed=0):
    """Synthetic kidney-like volumes for the smoke test: ellipsoid kidney, bright 'tumours', dark 'cysts'."""
    rng = np.random.default_rng(seed)
    for c in range(n_cases):
        Z, H, W = 24, 192, 192
        im = rng.normal(40, 15, (Z, H, W)).astype(np.float32)
        sg = np.zeros((Z, H, W), np.uint8)
        zz, yy, xx = np.mgrid[:Z, :H, :W]
        k = ((zz - 12) / 9) ** 2 + ((yy - 96) / 50) ** 2 + ((xx - 96) / 30) ** 2 < 1
        im[k] += 120; sg[k] = 1
        for _ in range(rng.integers(1, 4)):
            cz, cy, cx = rng.integers(6, 18), rng.integers(70, 120), rng.integers(80, 110)
            r = rng.uniform(2, 8); cls = rng.choice([2, 3])
            b = ((zz - cz) / max(r / 2.5, 1)) ** 2 + ((yy - cy) / r) ** 2 + ((xx - cx) / r) ** 2 < 1
            im[b] = rng.normal(220 if cls == 2 else 10, 10, b.sum()); sg[b] = cls
        key = f"case_{c:05d}_s0"
        np.save(os.path.join(out_dir, key + "_img.npy"), im.astype(np.int16))
        np.save(os.path.join(out_dir, key + "_seg.npy"), sg)
        json.dump(dict(case=f"case_{c:05d}", key=key, spacing=[2.5, 1.0, 1.0]), open(os.path.join(out_dir, key + "_meta.json"), "w"))
        open(os.path.join(out_dir, f"case_{c:05d}.done"), "w").write(json.dumps([key]))

def build_kssd_cache(cfg):
    """KSSD2025: 838 axial TIFF image/mask pairs. Stored as Z=1 'volumes'. No patient IDs exist."""
    import cv2
    files = sorted(glob.glob(cfg["KSSD_GLOB"], recursive=True))
    imgs = [f for f in files if "/label" not in f.lower() and "mask" not in os.path.basename(f).lower()]
    n = 0
    for f in imgs:
        cand = [f.replace("/image/", "/label/"), f.replace("/images/", "/labels/")]
        mf = next((m for m in cand if m != f and os.path.exists(m)), None)
        if mf is None:
            continue
        im = cv2.imread(f, cv2.IMREAD_UNCHANGED); mk = cv2.imread(mf, cv2.IMREAD_UNCHANGED)
        if im is None or mk is None:
            continue
        if im.ndim == 3: im = cv2.cvtColor(im, cv2.COLOR_BGR2GRAY)
        if mk.ndim == 3: mk = mk[..., 0]
        key = f"kssd_{n:04d}_s0"
        np.save(os.path.join(cfg["DATA_DIR"], key + "_img.npy"), im[None].astype(np.int16))
        np.save(os.path.join(cfg["DATA_DIR"], key + "_seg.npy"), (mk[None] > 0).astype(np.uint8))
        json.dump(dict(case=f"kssd_{n:04d}", key=key, spacing=[1.0, 1.0, 1.0], src=f), open(os.path.join(cfg["DATA_DIR"], key + "_meta.json"), "w"))
        open(os.path.join(cfg["DATA_DIR"], f"kssd_{n:04d}.done"), "w").write(json.dumps([key]))
        n += 1
    print(f"KSSD2025 cached: {n} image/mask pairs")

if CFG["SMOKE_TEST"]:
    make_synthetic_data(CFG["DATA_DIR"])
elif CFG["DATASET"] == "kssd2025":
    if not glob.glob(os.path.join(CFG["DATA_DIR"], "*.done")):
        build_kssd_cache(CFG)
elif CFG["RUN_PREPROCESS"]:
    from concurrent.futures import ProcessPoolExecutor
    try:
        cases = list_kits_cases(CFG["HF_REPO"]); repo = CFG["HF_REPO"]
    except Exception as e:
        print("primary repo failed:", e); repo = CFG["HF_REPO_FALLBACK"]; cases = list_kits_cases(repo)
    if CFG["N_CASES"]:
        cases = cases[: CFG["N_CASES"]]
    print(f"{len(cases)} cases from {repo}")
    t0 = time.time()
    with ProcessPoolExecutor(CFG["PREP_WORKERS"]) as ex:
        for i, (cid, status) in enumerate(ex.map(_process_case, [(c, repo, CFG) for c in cases])):
            if status != "skip" and (i % 10 == 0 or status.startswith("ERROR")):
                print(f"[{i+1}/{len(cases)}] {cid}: {status}  ({(time.time()-t0)/60:.1f} min)")
    print("preprocessing finished")

In [ ]:
# ===== Cell 4: patient-level folds =====
def load_index(cfg):
    metas = [json.load(open(p)) for p in sorted(glob.glob(os.path.join(cfg["DATA_DIR"], "*_meta.json")))]
    cases = sorted({m["case"] for m in metas})
    return metas, cases

METAS, CASES = load_index(CFG)
fold_file = os.path.join(CFG["OUT_DIR"], f'folds_{CFG["DATASET"]}_{CFG["N_FOLDS"]}_seed{CFG["SEED"]}.json')  # DATA_DIR may be read-only
if os.path.exists(fold_file) and set(json.load(open(fold_file))["cases"]) == set(CASES):
    FOLDS = json.load(open(fold_file))["folds"]
else:
    rng = np.random.default_rng(CFG["SEED"]); perm = list(rng.permutation(CASES))
    FOLDS = [sorted(perm[i::CFG["N_FOLDS"]]) for i in range(CFG["N_FOLDS"])]
    json.dump(dict(cases=CASES, folds=FOLDS), open(fold_file, "w"))
val_cases = set(FOLDS[CFG["FOLD"]])
TRAIN_KEYS = [m["key"] for m in METAS if m["case"] not in val_cases]
VAL_KEYS = [m["key"] for m in METAS if m["case"] in val_cases]
SPACING = {m["key"]: m["spacing"] for m in METAS}
if CFG["DATASET"] == "kssd2025":
    print("WARNING: KSSD2025 has no patient IDs; image-level folds can leak patients. Report as an upper bound.")
print(f"cases={len(CASES)}  train volumes={len(TRAIN_KEYS)}  val volumes={len(VAL_KEYS)}")

In [ ]:
# ===== Cell 5: dataset, augmentation, targets =====
import torch, cv2
from torch.utils.data import Dataset, DataLoader

def lowdose_noise(hu, dose, rng, sigma0=12.0):
    """Image-domain low-dose approximation: correlated Gaussian noise, std = sigma0*sqrt(1/dose - 1)."""
    if dose >= 0.999:
        return hu
    s = sigma0 * math.sqrt(1.0 / dose - 1.0)
    n = ndimage.gaussian_filter(rng.standard_normal(hu.shape).astype(np.float32), sigma=(0, 0.8, 0.8))
    n /= n.std() + 1e-6
    return hu + s * n

def lesion_instances(sem, lesion_classes, max_inst=24):
    masks, labels, centres = [], [], []
    for li, c in enumerate(lesion_classes):
        cc, n = ndimage.label(sem == c)
        for i in range(1, n + 1):
            m = cc == i
            if m.sum() < 1:
                continue
            dt = ndimage.distance_transform_edt(m)
            cy, cx = np.unravel_index(np.argmax(dt), dt.shape)   # most interior pixel
            masks.append(m.astype(np.uint8)); labels.append(li); centres.append((cy, cx, math.sqrt(m.sum() / math.pi)))
    if len(masks) > max_inst:
        order = np.argsort([-m.sum() for m in masks])[:max_inst]
        masks = [masks[i] for i in order]; labels = [labels[i] for i in order]; centres = [centres[i] for i in order]
    return masks, labels, centres

def centre_heatmap(centres, H, W, stride=2):
    h, w = H // stride, W // stride
    hm = np.zeros((h, w), np.float32)
    yy, xx = np.mgrid[:h, :w]
    for cy, cx, r in centres:
        py, px = min(int(cy // stride), h - 1), min(int(cx // stride), w - 1)
        sig = max(0.5 * r / stride, 1.0)
        g = np.exp(-((yy - py) ** 2 + (xx - px) ** 2) / (2 * sig ** 2))
        hm = np.maximum(hm, g); hm[py, px] = 1.0
    return hm

class SliceDS(Dataset):
    def __init__(self, keys, cfg, train):
        self.cfg, self.train, self.S = cfg, train, cfg["N_SLICES"]
        self.vols = []
        for k in keys:
            im = np.load(os.path.join(cfg["DATA_DIR"], k + "_img.npy"), mmap_mode="r")
            sg = np.load(os.path.join(cfg["DATA_DIR"], k + "_seg.npy"), mmap_mode="r")
            self.vols.append((im, sg))
        self.index, self.lesion_index = [], []
        les = np.array(cfg["LESION_CLASSES"])
        for v, (im, sg) in enumerate(self.vols):
            for z in range(im.shape[0]):
                self.index.append((v, z))
                if np.isin(sg[z], les).any():
                    self.lesion_index.append((v, z))
        self.rng = np.random.default_rng()

    def __len__(self):
        return self.cfg["ITERS_PER_EPOCH"] * self.cfg["BATCH"] if self.train else len(self.index)

    def _stack(self, v, z):
        im, sg = self.vols[v]
        idx = np.clip(np.arange(z - self.S // 2, z + self.S // 2 + 1), 0, im.shape[0] - 1)
        return np.asarray(im[idx], np.float32), np.asarray(sg[z], np.int64)

    def _augment(self, x, y):
        r = self.rng
        S, H, W = x.shape
        if r.random() < 0.5:
            x, y = x[:, :, ::-1], y[:, ::-1]
        if r.random() < 0.7:
            ang, sc = r.uniform(-15, 15), r.uniform(0.85, 1.15)
            M = cv2.getRotationMatrix2D((W / 2, H / 2), ang, sc); M[:, 2] += r.uniform(-8, 8, 2)
            x = np.stack([cv2.warpAffine(np.ascontiguousarray(c), M, (W, H), flags=cv2.INTER_LINEAR, borderValue=float(x.min())) for c in x])
            y = cv2.warpAffine(np.ascontiguousarray(y).astype(np.uint8), M, (W, H), flags=cv2.INTER_NEAREST, borderValue=0).astype(np.int64)
        if r.random() < 0.5:
            x = x * r.uniform(0.92, 1.08) + r.uniform(-self.cfg["INT_SHIFT"], self.cfg["INT_SHIFT"])
        if r.random() < self.cfg["LOWDOSE_P"]:
            x = lowdose_noise(x, r.uniform(0.1, 0.5), r, self.cfg["NOISE_SIGMA0"])
        if r.random() < self.cfg["BLUR_P"]:
            x = ndimage.gaussian_filter(x, sigma=(0, r.uniform(0.5, 1.0), r.uniform(0.5, 1.0)))
        return np.ascontiguousarray(x, np.float32), np.ascontiguousarray(y)

    def _crop(self, x, y, size):
        S, H, W = x.shape
        if H <= size and W <= size:
            return x, y
        les = np.argwhere(np.isin(y, self.cfg["LESION_CLASSES"]))
        if len(les) and self.rng.random() < 0.7:
            cy, cx = les[self.rng.integers(len(les))]
            y0 = int(np.clip(cy - self.rng.integers(size // 4, 3 * size // 4), 0, H - size))
            x0 = int(np.clip(cx - self.rng.integers(size // 4, 3 * size // 4), 0, W - size))
        else:
            y0, x0 = self.rng.integers(0, H - size + 1), self.rng.integers(0, W - size + 1)
        return x[:, y0:y0 + size, x0:x0 + size], y[y0:y0 + size, x0:x0 + size]

    def __getitem__(self, i):
        if self.train:
            pool = self.lesion_index if (self.lesion_index and self.rng.random() < self.cfg["LESION_SAMPLE_P"]) else self.index
            v, z = pool[self.rng.integers(len(pool))]
        else:
            v, z = self.index[i]
        x, y = self._stack(v, z)
        if self.train:
            if self.cfg["CROP"]:
                x, y = self._crop(x, y, self.cfg["CROP"])
            x, y = self._augment(x, y)
        masks, labels, centres = lesion_instances(y, self.cfg["LESION_CLASSES"])
        H, W = y.shape
        hm = centre_heatmap(centres, H, W)
        cidx = [min(int(c[0] // 2), H // 2 - 1) * (W // 2) + min(int(c[1] // 2), W // 2 - 1) for c in centres]
        return dict(x=torch.from_numpy(x), y=torch.from_numpy(y), hm=torch.from_numpy(hm)[None],
                    inst_masks=torch.from_numpy(np.stack(masks)) if masks else torch.zeros(0, H, W, dtype=torch.uint8),
                    inst_labels=torch.tensor(labels, dtype=torch.long), inst_cidx=torch.tensor(cidx, dtype=torch.long))

def collate(batch):
    out = {k: torch.stack([b[k] for b in batch]) for k in ("x", "y", "hm")}
    for k in ("inst_masks", "inst_labels", "inst_cidx"):
        out[k] = [b[k] for b in batch]
    return out

In [ ]:
# ===== Cell 6: model — PAWE, hybrid encoder (AIFI), U-Net decoder, LCQS, hierarchical query-graph decoder =====
import torch.nn as nn
import torch.nn.functional as F
import timm

def sincos_2d(h, w, dim, device):
    y, x = torch.meshgrid(torch.arange(h, device=device, dtype=torch.float32),
                          torch.arange(w, device=device, dtype=torch.float32), indexing="ij")
    return sincos_points(torch.stack([(x.flatten() + 0.5) / w, (y.flatten() + 0.5) / h], -1)[None], dim)

def sincos_points(p, dim, temp=10000.0):
    """p: (B, N, 2) normalised (x, y) in [0,1] -> (B, N, dim)."""
    d = dim // 4
    omega = 1.0 / temp ** (torch.arange(d, device=p.device, dtype=torch.float32) / d)
    ox = p[..., 0:1] * 2 * math.pi * 10 * omega; oy = p[..., 1:2] * 2 * math.pi * 10 * omega
    emb = torch.cat([ox.sin(), ox.cos(), oy.sin(), oy.cos()], -1)
    return F.pad(emb, (0, dim - emb.shape[-1]))

class PAWE(nn.Module):
    """Physics-aware multi-window enhancement: soft HU windows -> Zero-DCE-style per-pixel curves -> noise-gated residual."""
    def __init__(self, windows, n_slices, mode="pawe", n_iter=4, hidden=24):
        super().__init__()
        self.mode, self.S, self.n_iter = mode, n_slices, n_iter
        L = torch.tensor([w[0] for w in windows], dtype=torch.float32)
        Wd = torch.tensor([w[1] for w in windows], dtype=torch.float32)
        self.K = len(windows)
        if mode in ("pawe", "pawe_nogate", "wso"):
            self.level = nn.Parameter(L); self.log_width = nn.Parameter(Wd.log())
        else:
            self.register_buffer("level", L); self.register_buffer("log_width", Wd.log())
        if mode in ("pawe", "pawe_nogate"):
            C = self.K * n_slices
            self.curve = nn.Sequential(nn.Conv2d(C, hidden, 3, padding=1), nn.ReLU(True),
                                       nn.Conv2d(hidden, hidden, 3, padding=1), nn.ReLU(True),
                                       nn.Conv2d(hidden, self.K * n_iter, 3, padding=1), nn.Tanh())
            nn.init.zeros_(self.curve[-2].weight); nn.init.zeros_(self.curve[-2].bias)   # identity at start
            self.dn = nn.Conv2d(C, C, 3, padding=1, groups=C); nn.init.zeros_(self.dn.weight); nn.init.zeros_(self.dn.bias)
            self.gate = nn.Conv2d(C, C, 1)
        self.out_ch = self.K * n_slices

    def forward(self, hu):
        with torch.autocast(device_type=hu.device.type, enabled=False):
            hu = hu.float()
            B, S, H, W = hu.shape
            L = self.level.view(1, -1, 1, 1, 1); Wd = self.log_width.clamp(math.log(20.0), math.log(4000.0)).exp().view(1, -1, 1, 1, 1)
            y = torch.sigmoid(8.0 * (hu[:, None] - L) / Wd)                     # (B,K,S,H,W)
            alpha = None
            if self.mode in ("pawe", "pawe_nogate"):
                a = self.curve(y.reshape(B, -1, H, W)).view(B, self.n_iter, self.K, 1, H, W)
                for n in range(self.n_iter):
                    y = y + a[:, n] * y * (1 - y)                                 # LE-curve, stays in [0,1]
                f = y.reshape(B, -1, H, W)
                if self.mode == "pawe":
                    mu = F.avg_pool2d(f, 5, 1, 2); sd = (F.avg_pool2d(f * f, 5, 1, 2) - mu * mu).clamp_min(0).sqrt()
                    f = f + torch.sigmoid(self.gate(sd * 10.0)) * self.dn(f)      # noise-gated residual
                alpha = a
            else:
                f = y.reshape(B, -1, H, W)
            return (f - 0.5) / 0.25, alpha, y

class AIFI(nn.Module):
    def __init__(self, c_in, d, nhead=8):
        super().__init__()
        self.proj = nn.Conv2d(c_in, d, 1)
        self.layer = nn.TransformerEncoderLayer(d, nhead, 4 * d, dropout=0.0, batch_first=True, norm_first=True)
    def forward(self, f):
        x = self.proj(f); B, C, H, W = x.shape
        t = x.flatten(2).transpose(1, 2)
        t = self.layer(t + sincos_2d(H, W, C, x.device))
        return t.transpose(1, 2).reshape(B, C, H, W)

def cbr(i, o):
    return nn.Sequential(nn.Conv2d(i, o, 3, padding=1, bias=False), nn.BatchNorm2d(o), nn.ReLU(True))

class Up(nn.Module):
    def __init__(self, i, s, o):
        super().__init__(); self.c = nn.Sequential(cbr(i + s, o), cbr(o, o))
    def forward(self, x, skip=None):
        x = F.interpolate(x, scale_factor=2, mode="bilinear", align_corners=False)
        if skip is not None:
            if skip.shape[-2:] != x.shape[-2:]:
                x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
            x = torch.cat([x, skip], 1)
        return self.c(x)

class SAGE(nn.Module):
    """Mean-aggregator GraphSAGE layer on a dense, row-normalised adjacency."""
    def __init__(self, d):
        super().__init__(); self.norm = nn.LayerNorm(2 * d)
        self.mlp = nn.Sequential(nn.Linear(2 * d, d), nn.GELU(), nn.Linear(d, d))
    def forward(self, h, A):
        return h + self.mlp(self.norm(torch.cat([h, torch.bmm(A, h)], -1)))

class DecLayer(nn.Module):
    def __init__(self, d, graph_mode, nhead=8):
        super().__init__()
        self.graph_mode = graph_mode
        if graph_mode == "graph":
            self.sage = SAGE(d)
        elif graph_mode == "selfattn":
            self.n_sa = nn.LayerNorm(d); self.sa = nn.MultiheadAttention(d, nhead, batch_first=True)
        self.n_ca = nn.LayerNorm(d); self.ca = nn.MultiheadAttention(d, nhead, batch_first=True)
        self.n_ff = nn.LayerNorm(d); self.ff = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Linear(4 * d, d))
    def forward(self, nodes, qpos, K, mem, mpos, A):
        if self.graph_mode == "graph":
            nodes = self.sage(nodes, A)
        q = nodes[:, :K]
        if self.graph_mode == "selfattn":
            t = self.n_sa(q); q = q + self.sa(t + qpos, t + qpos, t)[0]
        t = self.n_ca(q); q = q + self.ca(t + qpos, mem + mpos, mem)[0]
        q = q + self.ff(self.n_ff(q))
        return torch.cat([q, nodes[:, K:]], 1)

def build_graph(qp, knn=6, tau=0.08, grid=4):
    """Three-level graph: K lesion queries, grid*grid region nodes, 1 kidney node. Returns row-normalised (B,N,N)."""
    B, K, _ = qp.shape; R = grid * grid; N = K + R + 1; dev = qp.device
    A = torch.zeros(B, N, N, device=dev)
    d = torch.cdist(qp, qp) + torch.eye(K, device=dev)[None] * 1e3
    kk = min(knn, K - 1)
    if kk > 0:
        dv, di = d.topk(kk, dim=-1, largest=False)
        A[:, :K, :K].scatter_(2, di, torch.exp(-dv ** 2 / (2 * tau ** 2)))
        A[:, :K, :K] = torch.maximum(A[:, :K, :K], A[:, :K, :K].transpose(1, 2))
    g = (torch.arange(grid, device=dev, dtype=torch.float32) + 0.5) / grid
    gy, gx = torch.meshgrid(g, g, indexing="ij"); rc = torch.stack([gx.flatten(), gy.flatten()], -1)
    dqr = torch.cdist(qp, rc[None].expand(B, -1, -1))
    _, ri = dqr.topk(2, dim=-1, largest=False)
    qr = torch.zeros(B, K, R, device=dev).scatter_(2, ri, 1.0)
    A[:, :K, K:K + R] = qr; A[:, K:K + R, :K] = qr.transpose(1, 2)
    rr = (torch.cdist(rc, rc) < (1.0 / grid) * 1.01).float() - torch.eye(R, device=dev)
    A[:, K:K + R, K:K + R] = rr
    A[:, -1, :-1] = 1.0; A[:, :-1, -1] = 1.0
    return A / A.sum(-1, keepdim=True).clamp_min(1e-6)

class MWHQNet(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.C = len(cfg["CLASSES"]); self.L = len(cfg["LESION_CLASSES"])
        wins = [cfg["SINGLE_WINDOW"]] if cfg["ENHANCE"] == "single" else cfg["WINDOWS"]
        mode = {"pawe": "pawe", "pawe_nogate": "pawe_nogate", "wso": "wso"}.get(cfg["ENHANCE"], "fixed")
        self.pawe = PAWE(wins, cfg["N_SLICES"], mode=mode)
        try:
            self.enc = timm.create_model("resnet34", features_only=True, pretrained=cfg["PRETRAINED"], in_chans=self.pawe.out_ch)
        except Exception as e:
            print("pretrained weights unavailable, training from scratch:", e)
            self.enc = timm.create_model("resnet34", features_only=True, pretrained=False, in_chans=self.pawe.out_ch)
        ch = self.enc.feature_info.channels()          # [64, 64, 128, 256, 512] at strides 2..32
        D = cfg["D_MODEL"]
        self.top = AIFI(ch[4], D) if cfg["AIFI"] else nn.Conv2d(ch[4], D, 1)
        self.u16 = Up(D, ch[3], 128); self.u8 = Up(128, ch[2], 96); self.u4 = Up(96, ch[1], 64)
        self.u2 = Up(64, ch[0], 48); self.u1 = Up(48, 0, 32)
        self.dense_head = nn.Conv2d(32, self.C, 1)
        if cfg["QUERY"]:
            self.hm_head = nn.Sequential(cbr(48, 48), nn.Conv2d(48, 1, 1)); nn.init.constant_(self.hm_head[-1].bias, -2.19)
            self.pix = nn.Sequential(cbr(48, cfg["MASK_DIM"]), nn.Conv2d(cfg["MASK_DIM"], cfg["MASK_DIM"], 1))
            self.mem16 = nn.Conv2d(128, D, 1); self.mem8 = nn.Conv2d(96, D, 1)
            self.lvl = nn.Parameter(torch.zeros(2, D))
            self.q_in = nn.Sequential(nn.Linear(48 + 96 + self.pawe.K, D), nn.GELU(), nn.Linear(D, D))
            self.q_pos = nn.Sequential(nn.Linear(D, D), nn.GELU(), nn.Linear(D, D))
            self.region_proj = nn.Linear(D, D); self.kidney_proj = nn.Linear(D, D)
            self.layers = nn.ModuleList([DecLayer(D, cfg["GRAPH"]) for _ in range(cfg["N_DEC_LAYERS"])])
            self.out_norm = nn.LayerNorm(D)
            self.cls_head = nn.Linear(D, self.L + 1)
            self.mask_head = nn.Sequential(nn.Linear(D, D), nn.GELU(), nn.Linear(D, cfg["MASK_DIM"]))

    def _heads(self, q, pix):
        h = self.out_norm(q)
        return self.cls_head(h), torch.einsum("bkd,bdhw->bkhw", self.mask_head(h), pix)

    def forward(self, x, gt_cidx=None, tf_p=0.0):
        B, S, H, W = x.shape
        f, alpha, ywin = self.pawe(x)
        e = self.enc(f)
        p32 = self.top(e[4]); p16 = self.u16(p32, e[3]); p8 = self.u8(p16, e[2]); p4 = self.u4(p8, e[1]); p2 = self.u2(p4, e[0])
        p1 = self.u1(p2)
        if p1.shape[-2:] != (H, W):
            p1 = F.interpolate(p1, size=(H, W), mode="bilinear", align_corners=False)
        out = dict(dense=self.dense_head(p1), alpha=alpha)
        if not self.cfg["QUERY"]:
            return out
        K = self.cfg["N_QUERIES"]
        hm_logit = self.hm_head(p2); out["hm"] = hm_logit
        h2, w2 = p2.shape[-2:]
        with torch.no_grad():
            hm = hm_logit.float().sigmoid()
            peaks = hm * (F.max_pool2d(hm, 3, 1, 1) == hm).float()
            idx = peaks.flatten(1).topk(K, dim=1).indices                      # (B,K) lesion-centre query selection
            if self.training and gt_cidx is not None and tf_p > 0:
                idx = idx.clone()
                for b in range(B):
                    g = gt_cidx[b]
                    if len(g) and random.random() < tf_p:
                        g = g[: K // 2].to(idx.device)
                        keep = idx[b][~torch.isin(idx[b], g)][: K - len(g)]
                        idx[b] = torch.cat([keep, g])[:K]
        ys, xs = (idx // w2).float(), (idx % w2).float()
        qp = torch.stack([(xs + 0.5) / w2, (ys + 0.5) / h2], -1)              # (B,K,2) in [0,1]
        f2 = torch.gather(p2.flatten(2), 2, idx[:, None].expand(-1, p2.shape[1], -1)).transpose(1, 2)
        f8 = F.grid_sample(p8, (qp * 2 - 1)[:, :, None], align_corners=False)[..., 0].transpose(1, 2)
        yc = ywin[:, :, S // 2]                                                # (B,Kw,H,W) centre-slice window maps
        yc = F.adaptive_avg_pool2d(F.avg_pool2d(yc.float(), 5, 1, 2), (h2, w2))
        fw = torch.gather(yc.flatten(2), 2, idx[:, None].expand(-1, yc.shape[1], -1)).transpose(1, 2)
        q = self.q_in(torch.cat([f2, f8, fw.to(f2.dtype)], -1))
        qpos = self.q_pos(sincos_points(qp, q.shape[-1]).to(q.dtype))
        m16 = self.mem16(p16); m8 = self.mem8(p8)
        mem = torch.cat([m16.flatten(2).transpose(1, 2), m8.flatten(2).transpose(1, 2)], 1)
        mpos = torch.cat([sincos_2d(*m16.shape[-2:], m16.shape[1], x.device) + self.lvl[0],
                          sincos_2d(*m8.shape[-2:], m8.shape[1], x.device) + self.lvl[1]], 1).to(mem.dtype)
        region = self.region_proj(F.adaptive_avg_pool2d(m16, 4).flatten(2).transpose(1, 2))
        with torch.no_grad():
            fgp = 1 - out["dense"].float().softmax(1)[:, :1]
            wk = F.adaptive_avg_pool2d(fgp, m16.shape[-2:]).flatten(1)
            wk = (wk / wk.sum(1, keepdim=True).clamp_min(1e-6)).to(m16.dtype)
        kidney = self.kidney_proj(torch.bmm(m16.flatten(2), wk[..., None]).transpose(1, 2))
        nodes = torch.cat([q, region, kidney], 1)
        A = build_graph(qp.float(), self.cfg["KNN"]).to(nodes.dtype) if self.cfg["GRAPH"] == "graph" else None
        pix = self.pix(p2)
        aux = []
        for layer in self.layers:
            nodes = layer(nodes, qpos, K, mem, mpos, A)
            aux.append(self._heads(nodes[:, :K], pix))
        out.update(cls=aux[-1][0], masks=aux[-1][1], aux=aux, qp=qp)
        return out

def fuse(out, cfg):
    """Final per-pixel class probabilities: dense softmax, lesion channels blended with query masks."""
    pd = out["dense"].float().softmax(1)
    if not (cfg["QUERY"] and cfg["FUSION"]):
        return pd
    H, W = pd.shape[-2:]
    pc = out["cls"].float().softmax(-1)[..., :-1]                               # (B,K,L)
    pm = F.interpolate(out["masks"].float(), size=(H, W), mode="bilinear", align_corners=False).sigmoid()
    pd = pd.clone(); b = cfg["FUSION_BETA"]
    for j, c in enumerate(cfg["LESION_CLASSES"]):
        pq = (pc[:, :, j, None, None] * pm).amax(1)
        pd[:, c] = (1 - b) * pd[:, c] + b * pq
    return pd

In [ ]:
# ===== Cell 7: losses — dense Dice+CE, centre focal, Hungarian query loss, curve smoothness =====
from scipy.optimize import linear_sum_assignment

def dense_loss(logits, y, C, w):
    logits = logits.float()
    ce = F.cross_entropy(logits, y, weight=torch.tensor(w, device=logits.device, dtype=torch.float32))
    p = logits.softmax(1); oh = F.one_hot(y, C).permute(0, 3, 1, 2).float()
    inter = (p * oh).sum((0, 2, 3)); den = p.sum((0, 2, 3)) + oh.sum((0, 2, 3))
    dice = 1 - ((2 * inter + 1) / (den + 1))[1:].mean()
    return ce + dice

def focal_hm(logit, gt):
    p = logit.float().sigmoid().clamp(1e-4, 1 - 1e-4)
    pos = gt.eq(1).float(); neg = 1 - pos
    lp = -(torch.log(p) * (1 - p) ** 2 * pos).sum()
    ln = -(torch.log(1 - p) * p ** 2 * (1 - gt) ** 4 * neg).sum()
    return (lp + ln) / pos.sum().clamp_min(1)

@torch.no_grad()
def hungarian(cls_logit, mask_logit, gl, gm_lr):
    if len(gl) == 0:
        return np.array([], int), np.array([], int)
    prob = cls_logit.float().softmax(-1)[:, gl]
    x = mask_logit.float().flatten(1); g = gm_lr.flatten(1)
    hw = x.shape[1]
    bce = (F.softplus(-x) @ g.T + F.softplus(x) @ (1 - g).T) / hw
    s = x.sigmoid()
    dice = 1 - (2 * s @ g.T + 1) / (s.sum(1)[:, None] + g.sum(1)[None] + 1)
    cost = -2 * prob + 5 * bce + 5 * dice
    return linear_sum_assignment(cost.cpu().numpy())

def query_loss(out, batch, cfg):
    Lc = len(cfg["LESION_CLASSES"]); dev = out["dense"].device
    H, W = batch["y"].shape[-2:]
    total = 0.0; n_gt = max(sum(len(l) for l in batch["inst_labels"]), 1)
    for cls_logit, mask_logit in out["aux"]:
        B, K, _ = cls_logit.shape
        tgt = torch.full((B, K), Lc, dtype=torch.long, device=dev)
        lb, ld = 0.0, 0.0
        h2, w2 = mask_logit.shape[-2:]
        for b in range(B):
            gl = batch["inst_labels"][b].to(dev)
            if len(gl) == 0:
                continue
            gm = batch["inst_masks"][b].to(dev).float()
            gm_lr = F.interpolate(gm[None], size=(h2, w2), mode="area")[0]
            qi, gi = hungarian(cls_logit[b], mask_logit[b], gl, gm_lr)
            qi = torch.as_tensor(qi, device=dev); gi = torch.as_tensor(gi, device=dev)
            tgt[b, qi] = gl[gi]
            pm = F.interpolate(mask_logit[b, qi][None].float(), size=(H, W), mode="bilinear", align_corners=False)[0]
            gt = gm[gi]
            lb = lb + F.binary_cross_entropy_with_logits(pm, gt, reduction="none").mean((1, 2)).sum()
            s = pm.sigmoid()
            ld = ld + (1 - (2 * (s * gt).sum((1, 2)) + 1) / (s.sum((1, 2)) + gt.sum((1, 2)) + 1)).sum()
        wcls = torch.ones(Lc + 1, device=dev); wcls[-1] = 0.1
        lcls = F.cross_entropy(cls_logit.float().flatten(0, 1), tgt.flatten(), weight=wcls)
        total = total + 2 * lcls + (5 * lb + 5 * ld) / n_gt
    return total / len(out["aux"])

def tv_loss(alpha):
    if alpha is None:
        return 0.0
    a = alpha.float()
    return (a[..., 1:, :] - a[..., :-1, :]).abs().mean() + (a[..., :, 1:] - a[..., :, :-1]).abs().mean()

def total_loss(out, batch, cfg):
    y = batch["y"]
    l = dense_loss(out["dense"], y, len(cfg["CLASSES"]), cfg["CE_WEIGHTS"])
    parts = dict(dense=float(l))
    if cfg["QUERY"]:
        lh = focal_hm(out["hm"], batch["hm"]); lq = query_loss(out, batch, cfg)
        l = l + cfg["W_HM"] * lh + cfg["W_QUERY"] * lq
        parts.update(hm=float(lh), query=float(lq))
    if out["alpha"] is not None:
        l = l + cfg["W_TV"] * tv_loss(out["alpha"])
    return l, parts

In [ ]:
# ===== Cell 8: 3D inference, post-processing, metrics =====
@torch.no_grad()
def predict_volume(model, img, cfg, dose=1.0, bs=16, tta=True, seed=0):
    dev = next(model.parameters()).device
    model.eval()
    rng = np.random.default_rng(seed)
    vol = np.asarray(img, np.float32)
    if dose < 1.0:
        vol = lowdose_noise(vol, dose, rng, cfg["NOISE_SIGMA0"])
    Z = vol.shape[0]; S = cfg["N_SLICES"]
    probs = np.zeros((len(cfg["CLASSES"]),) + vol.shape, np.float16)
    for z0 in range(0, Z, bs):
        zs = list(range(z0, min(Z, z0 + bs)))
        xb = np.stack([vol[np.clip(np.arange(z - S // 2, z + S // 2 + 1), 0, Z - 1)] for z in zs])
        xt = torch.from_numpy(xb).to(dev)
        with torch.autocast(device_type=dev.type, enabled=cfg["AMP"] and dev.type == "cuda"):
            p = fuse(model(xt), cfg)
            if tta:
                p = 0.5 * (p + fuse(model(xt.flip(-1)), cfg).flip(-1))
        probs[:, z0:z0 + len(zs)] = p.float().permute(1, 0, 2, 3).cpu().numpy()
    return probs

def postprocess(probs, spacing, cfg):
    lab = probs.argmax(0).astype(np.uint8)
    vox = float(np.prod(spacing))
    non_les = [c for c in range(len(cfg["CLASSES"])) if c not in cfg["LESION_CLASSES"]]
    fallback = np.array(non_les)[np.argmax(probs[non_les], 0)]
    gate = None
    if cfg["KIDNEY_CLASS"] is not None:
        organ = lab == cfg["KIDNEY_CLASS"]
        cc_k, n_k = ndimage.label(organ)
        if n_k:  # kidney components >= 2 ml (drops specks; keeps a partially visible second kidney)
            sizes = ndimage.sum(organ, cc_k, range(1, n_k + 1)) * vox
            big = np.isin(cc_k, 1 + np.where(sizes >= 2000)[0])
            organ = big if big.any() else organ
        gate = ndimage.binary_dilation(organ, iterations=3)
    for c in cfg["LESION_CLASSES"]:
        cc, n = ndimage.label(lab == c)
        if n == 0:
            continue
        ids = np.arange(1, n + 1)
        size = ndimage.sum(np.ones_like(cc), cc, ids) * vox
        pmax = ndimage.maximum(probs[c].astype(np.float32), cc, ids)
        bad = (size < cfg["MIN_LESION_MM3"]) | (pmax < cfg["TAU_HIGH"])
        if gate is not None:
            touch = ndimage.maximum(gate.astype(np.uint8), cc, ids) > 0
            bad |= ~touch
        rm = np.isin(cc, ids[bad])
        lab[rm] = fallback[rm]
    return lab

def _surface(m):
    st = ndimage.generate_binary_structure(m.ndim, 1)
    if m.shape[0] == 1:          # 2D data stored as Z=1: erode in-plane only
        st[0] = False; st[2] = False
    return m & ~ndimage.binary_erosion(m, structure=st, border_value=1)

def dice(a, b):
    sa, sb = a.sum(), b.sum()
    return 1.0 if sa + sb == 0 else 2.0 * (a & b).sum() / (sa + sb)

def surf_metrics(pred, gt, spacing, tol=2.0):
    if pred.sum() == 0 or gt.sum() == 0:
        return np.nan, np.nan
    sp, sg = _surface(pred), _surface(gt)
    dg = ndimage.distance_transform_edt(~sg, sampling=spacing); dp = ndimage.distance_transform_edt(~sp, sampling=spacing)
    d1, d2 = dg[sp], dp[sg]
    hd95 = float(np.percentile(np.concatenate([d1, d2]), 95))
    nsd = float(((d1 <= tol).sum() + (d2 <= tol).sum()) / (len(d1) + len(d2)))
    return hd95, nsd

def hec_masks(lab, cfg):
    if cfg["DATASET"] == "kits23":
        return {"kidney+masses": lab > 0, "masses": np.isin(lab, [2, 3]), "tumor": lab == 2, "cyst": lab == 3}
    return {"stone": lab == 1}

def lesion_detection(pred, gt, spacing, cfg, key):
    rows = []
    vox = float(np.prod(spacing))
    gt_les = np.isin(gt, cfg["LESION_CLASSES"]); pr_les = np.isin(pred, cfg["LESION_CLASSES"])
    for c in cfg["LESION_CLASSES"]:
        cc, n = ndimage.label(gt == c)
        for i in range(1, n + 1):
            m = cc == i; vol = m.sum() * vox
            diam = 2 * math.sqrt(vol / math.pi) if gt.shape[0] == 1 else (6 * vol / math.pi) ** (1 / 3)
            rows.append(dict(key=key, cls=cfg["CLASSES"][c], diam_mm=diam,
                             hit_any=bool((pr_les & m).any()), hit_cls=bool(((pred == c) & m).any())))
    pcc, pn = ndimage.label(pr_les)
    fp = sum(1 for i in range(1, pn + 1) if not (gt_les & (pcc == i)).any())
    return rows, fp

def evaluate(model, keys, cfg, dose=1.0, tta=True, max_vols=None):
    case_rows, les_rows = [], []
    for key in keys[: max_vols or len(keys)]:
        img = np.load(os.path.join(cfg["DATA_DIR"], key + "_img.npy"))
        gt = np.load(os.path.join(cfg["DATA_DIR"], key + "_seg.npy"))
        sp = SPACING[key]
        lab = postprocess(predict_volume(model, img, cfg, dose=dose, tta=tta), sp, cfg)
        r = dict(key=key, case=key.rsplit("_s", 1)[0], dose=dose)
        P, G = hec_masks(lab, cfg), hec_masks(gt, cfg)
        for name in G:
            r[f"dice_{name}"] = dice(P[name], G[name])
            r[f"hd95_{name}"], r[f"nsd_{name}"] = surf_metrics(P[name], G[name], sp)
            r[f"tp_{name}"] = int((P[name] & G[name]).sum()); r[f"ps_{name}"] = int(P[name].sum()); r[f"gs_{name}"] = int(G[name].sum())
        lr, fp = lesion_detection(lab, gt, sp, cfg, key)
        r["fp_lesions"] = fp
        case_rows.append(r); les_rows += lr
    import pandas as pd
    return pd.DataFrame(case_rows), pd.DataFrame(les_rows)

def size_bins(cfg):
    return [0, 3, 6, 20, 1e9] if cfg["DATASET"] != "kits23" else [0, 10, 20, 40, 1e9]

def summarise(df_case, df_les, cfg):
    import pandas as pd
    s = {}
    G = list(hec_masks(np.zeros(1, np.uint8), cfg).keys())
    # case-level Dice: pool both kidney ROIs of a case via confusion counts
    agg = df_case.groupby("case")[[c for c in df_case if c.startswith(("tp_", "ps_", "gs_"))]].sum()
    for g in G:
        d = np.where(agg[f"ps_{g}"] + agg[f"gs_{g}"] > 0, 2 * agg[f"tp_{g}"] / (agg[f"ps_{g}"] + agg[f"gs_{g}"]).clip(lower=1), 1.0)
        s[f"Dice {g}"] = float(np.mean(d))
        s[f"NSD@2mm {g}"] = float(df_case[f"nsd_{g}"].mean())
        s[f"HD95 {g} (mm)"] = float(df_case[f"hd95_{g}"].mean())
    s["FP lesions / kidney ROI"] = float(df_case["fp_lesions"].mean())
    if len(df_les):
        b = size_bins(cfg)
        df_les = df_les.assign(bin=pd.cut(df_les["diam_mm"], b, right=False))
        for k, g in df_les.groupby("bin", observed=True):
            s[f"Sens {k} (n={len(g)})"] = float(g["hit_any"].mean())
        s["Sens all lesions"] = float(df_les["hit_any"].mean())
    return s

In [ ]:
# ===== Cell 9: training loop (resumable) =====
dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MWHQNet(CFG).to(dev)
n_par = sum(p.numel() for p in model.parameters()) / 1e6
print(f"device={dev}  params={n_par:.1f}M")

train_ds = SliceDS(TRAIN_KEYS, CFG, train=True)
train_dl = DataLoader(train_ds, batch_size=CFG["BATCH"], shuffle=False, num_workers=CFG["NUM_WORKERS"],
                      collate_fn=collate, pin_memory=True, drop_last=True,
                      persistent_workers=CFG["NUM_WORKERS"] > 0,
                      worker_init_fn=lambda w: setattr(torch.utils.data.get_worker_info().dataset, "rng", np.random.default_rng(np.random.randint(1 << 30) + w)))
pawe_params = [p for n, p in model.named_parameters() if n.startswith("pawe.")]
other = [p for n, p in model.named_parameters() if not n.startswith("pawe.")]
opt = torch.optim.AdamW([{"params": other}, {"params": pawe_params, "lr": CFG["LR"] * 3, "weight_decay": 0.0}],
                        lr=CFG["LR"], weight_decay=CFG["WD"])
total_steps = CFG["EPOCHS"] * CFG["ITERS_PER_EPOCH"]; warm = CFG["WARMUP_EPOCHS"] * CFG["ITERS_PER_EPOCH"]
sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0, (s + 1) / max(warm, 1)) * 0.5 * (1 + math.cos(math.pi * min(s, total_steps) / total_steps)))
scaler = torch.amp.GradScaler("cuda", enabled=CFG["AMP"] and dev.type == "cuda")
start_ep, best = 0, -1.0
ck_last, ck_best = os.path.join(RUN_DIR, "last.pt"), os.path.join(RUN_DIR, "best.pt")
if CFG["RESUME"] and os.path.exists(ck_last):
    ck = torch.load(ck_last, map_location=dev, weights_only=False)
    model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"]); sched.load_state_dict(ck["sched"]); scaler.load_state_dict(ck["scaler"])
    start_ep, best = ck["epoch"] + 1, ck["best"]
    print(f"resumed from epoch {start_ep}, best={best:.4f}")
log_path = os.path.join(RUN_DIR, "log.jsonl")
SEL_METRIC = "Dice masses" if CFG["DATASET"] == "kits23" else "Dice stone"

for ep in range(start_ep, CFG["EPOCHS"]):
    model.train(); t0 = time.time(); run = {}
    tf_p = CFG["TF_P0"] * max(0.0, 1 - ep / (0.5 * CFG["EPOCHS"]))
    for it, batch in enumerate(train_dl):
        x = batch["x"].to(dev, non_blocking=True)
        batch["y"] = batch["y"].to(dev, non_blocking=True); batch["hm"] = batch["hm"].to(dev, non_blocking=True)
        with torch.autocast(device_type=dev.type, enabled=CFG["AMP"] and dev.type == "cuda"):
            out = model(x, gt_cidx=batch["inst_cidx"], tf_p=tf_p)
        loss, parts = total_loss(out, batch, CFG)
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), CFG["GRAD_CLIP"])
        scaler.step(opt); scaler.update(); sched.step()
        for k, v in parts.items():
            run[k] = run.get(k, 0) + v
        if it % 100 == 0:
            print(f"ep{ep} it{it} loss={loss.item():.3f} " + " ".join(f"{k}={v:.3f}" for k, v in parts.items()))
    rec = dict(epoch=ep, time_min=(time.time() - t0) / 60, **{k: v / (it + 1) for k, v in run.items()})
    if isinstance(model.pawe.level, nn.Parameter):
        rec["windows"] = [(round(float(l), 1), round(float(w), 1)) for l, w in zip(model.pawe.level, model.pawe.log_width.exp())]
    if (ep + 1) % CFG["VAL_EVERY"] == 0 or ep == CFG["EPOCHS"] - 1:
        dfc, dfl = evaluate(model, VAL_KEYS, CFG, tta=False, max_vols=None if not CFG["SMOKE_TEST"] else 2)
        summ = summarise(dfc, dfl, CFG); rec.update(summ)
        if summ[SEL_METRIC] > best:
            best = summ[SEL_METRIC]; torch.save(model.state_dict(), ck_best)
    rec["best"] = best
    torch.save(dict(model=model.state_dict(), opt=opt.state_dict(), sched=sched.state_dict(), scaler=scaler.state_dict(), epoch=ep, best=best), ck_last)
    open(log_path, "a").write(json.dumps(rec, default=str) + "\n")
    print(json.dumps(rec, default=str))
if not os.path.exists(ck_best):
    torch.save(model.state_dict(), ck_best)

In [ ]:
# ===== Cell 10: final evaluation (TTA) + low-dose robustness; saves CSVs for statistics =====
import pandas as pd
model.load_state_dict(torch.load(ck_best, map_location=dev, weights_only=False))
all_summ = {}
for d in CFG["TEST_DOSES"]:
    dfc, dfl = evaluate(model, VAL_KEYS, CFG, dose=d, tta=True, max_vols=None if not CFG["SMOKE_TEST"] else 2)
    dfc.to_csv(os.path.join(RUN_DIR, f"cases_dose{d}.csv"), index=False)
    dfl.to_csv(os.path.join(RUN_DIR, f"lesions_dose{d}.csv"), index=False)
    all_summ[f"dose={d}"] = summarise(dfc, dfl, CFG)
summary = pd.DataFrame(all_summ)
summary.to_csv(os.path.join(RUN_DIR, "summary.csv"))
print(summary.round(4).to_string())

In [ ]:
# ===== Cell 11: statistics — paired Wilcoxon + bootstrap CI between two runs (e.g. baseline vs MW-HQNet) =====
from scipy.stats import wilcoxon

def case_dice(df, g):
    a = df.groupby("case")[[f"tp_{g}", f"ps_{g}", f"gs_{g}"]].sum()
    den = a[f"ps_{g}"] + a[f"gs_{g}"]
    return pd.Series(np.where(den > 0, 2 * a[f"tp_{g}"] / den.clip(lower=1), 1.0), index=a.index)

def compare_runs(run_a, run_b, metric_group, dose=1.0, n_boot=2000, seed=0):
    A = case_dice(pd.read_csv(os.path.join(run_a, f"cases_dose{dose}.csv")), metric_group)
    B = case_dice(pd.read_csv(os.path.join(run_b, f"cases_dose{dose}.csv")), metric_group)
    idx = A.index.intersection(B.index); d = (B[idx] - A[idx]).values
    p = wilcoxon(d).pvalue if np.any(d != 0) else 1.0
    rng = np.random.default_rng(seed)
    boots = [rng.choice(d, len(d)).mean() for _ in range(n_boot)]
    return dict(n_cases=len(d), mean_diff=float(d.mean()), ci95=(float(np.percentile(boots, 2.5)), float(np.percentile(boots, 97.5))), wilcoxon_p=float(p))

# Example (after training a baseline run with QUERY=False, ENHANCE='single', AIFI=False, RUN_NAME='baseline_unet'):
# print(compare_runs(os.path.join(CFG['OUT_DIR'], 'baseline_unet_fold0'), RUN_DIR, 'masses'))
# Apply Holm correction across all comparisons you report.

In [ ]:
# ===== Cell 12: qualitative figure — input window, learned PAWE channels, GT vs prediction =====
import matplotlib.pyplot as plt
key = next((k for k in VAL_KEYS if np.isin(np.load(os.path.join(CFG["DATA_DIR"], k + "_seg.npy")), CFG["LESION_CLASSES"]).any()), VAL_KEYS[0])
img = np.load(os.path.join(CFG["DATA_DIR"], key + "_img.npy")); gt = np.load(os.path.join(CFG["DATA_DIR"], key + "_seg.npy"))
z = int(np.argmax(np.isin(gt, CFG["LESION_CLASSES"]).sum((1, 2))))
lab = postprocess(predict_volume(model, img, CFG), SPACING[key], CFG)
S = CFG["N_SLICES"]
x = torch.from_numpy(img[np.clip(np.arange(z - S // 2, z + S // 2 + 1), 0, img.shape[0] - 1)].astype(np.float32))[None].to(dev)
with torch.no_grad():
    _, _, ywin = model.pawe(x)
nw = ywin.shape[1]
fig, ax = plt.subplots(1, 3 + nw, figsize=(3.2 * (3 + nw), 3.4))
lo, hi = CFG["SINGLE_WINDOW"][0] - CFG["SINGLE_WINDOW"][1] / 2, CFG["SINGLE_WINDOW"][0] + CFG["SINGLE_WINDOW"][1] / 2
ax[0].imshow(img[z], cmap="gray", vmin=lo, vmax=hi); ax[0].set_title(f"{key} z={z}")
for k in range(nw):
    ax[1 + k].imshow(ywin[0, k, S // 2].float().cpu(), cmap="gray"); ax[1 + k].set_title(f"PAWE window {k}")
ax[1 + nw].imshow(gt[z], vmin=0, vmax=len(CFG["CLASSES"]) - 1, cmap="tab10", interpolation="nearest"); ax[1 + nw].set_title("ground truth")
ax[2 + nw].imshow(lab[z], vmin=0, vmax=len(CFG["CLASSES"]) - 1, cmap="tab10", interpolation="nearest"); ax[2 + nw].set_title("MW-HQNet")
for a in ax: a.axis("off")
plt.tight_layout(); plt.savefig(os.path.join(RUN_DIR, "qualitative.png"), dpi=200); plt.show()
if isinstance(model.pawe.level, nn.Parameter):
    print("learned windows (level, width):", [(round(float(l), 1), round(float(w), 1)) for l, w in zip(model.pawe.level, model.pawe.log_width.exp())])

## Ablation runs (one config change each, same fold and seed)
| RUN_NAME | changes |
|---|---|
| `baseline_unet` | `ENHANCE='single', AIFI=False, QUERY=False` |
| `abl_no_pawe` | `ENHANCE='fixed_multi'` |
| `abl_wso` | `ENHANCE='wso'` (learned windows only, the WSO-style baseline) |
| `abl_no_gate` | `ENHANCE='pawe_nogate'` |
| `abl_single_window` | `ENHANCE='single'` |
| `abl_no_aifi` | `AIFI=False` |
| `abl_no_query` | `QUERY=False` |
| `abl_selfattn` | `GRAPH='selfattn'` |
| `abl_no_interaction` | `GRAPH='none'` |
| `abl_no_fusion` | `FUSION=False` (evaluation-only; reuse the full model's `best.pt`) |

Then compare each against `mwhqnet_full` with Cell 11 and apply Holm correction.